In [ ]:
import pandas as pd

df = pd.read_csv("data_for_embeddings_part_1.csv")

In [ ]:
df = df['text']

In [ ]:
df.shape

(10000,)

In [ ]:
df = df.to_frame()
df.shape

(10000, 1)

In [ ]:
import pandas as pd

# Function to split text into sentences based on period (.)
def split_into_sentences(text):
    # Split by period and remove leading/trailing spaces from each sentence
    sentences = [sentence.strip() for sentence in text.split('.') if sentence]
    return sentences

# Apply function to the 'text' column and explode the list into separate rows
df_sentences = df['text'].apply(split_into_sentences).explode().reset_index(drop=True)

# Create new DataFrame with the sentences
df_sentences = pd.DataFrame({'sentence': df_sentences})

# Show the new DataFrame
df_sentences.head()


,sentence
0,"OPINION\nRABINOWITZ, Justice"
1,I
2,"INTRODUCTION\nThis appeal centers on the application of the notice requirement of the Differing Site Conditions clause found in a contract between appellant Neal & Company, Inc"
3,(NCI) and appellee City of Dillingham (City)
4,"Ap-pellee CH2M Hill (Hill), the City's engineer on the project, is involved in this appeal primarily because it acted as the City's representative on the project"


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

df_sentences.head(50)

,sentence
0,"OPINION\nRABINOWITZ, Justice"
1,I
2,"INTRODUCTION\nThis appeal centers on the application of the notice requirement of the Differing Site Conditions clause found in a contract between appellant Neal & Company, Inc"
3,(NCI) and appellee City of Dillingham (City)
4,"Ap-pellee CH2M Hill (Hill), the City's engineer on the project, is involved in this appeal primarily because it acted as the City's representative on the project"
5,"NCI claims that it encountered difficulties in excavation during the project because of unexpected soil conditions, that it gave notice of these unexpected conditions to Hill (and thus constructively to the City), and that therefore it is entitled to assert a claim under the Differing Site Conditions clause"
6,The City and Hill claim that no notice of a differing site condition was given
7,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition
8,"NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge"
9,II


In [ ]:
df_sentences.shape

(2731607, 1)

In [ ]:
import random
import pandas as pd

# Assuming df_sentences is your DataFrame with shape (1000, 1)
# and df_sentences['sentences'] contains the sentences.

def mask_sentence(sentence, mask_prob=0.1):
    words = sentence.split()  # Split sentence into words
    masked_words = []

    for word in words:
        if random.random() < mask_prob:  # 10% chance to mask the word
            masked_words.append("[MASK]")
        else:
            masked_words.append(word)

    return " ".join(masked_words)  # Join words back into a sentence

# Apply the masking function to each sentence in the DataFrame
df_sentences['masked_sentences'] = df_sentences['sentence'].apply(mask_sentence)

# Now df_sentences contains the original sentences and the masked versions


In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

df_sentences.head(50)

,sentence,masked_sentences
0,"OPINION\nRABINOWITZ, Justice",OPINION [MASK] [MASK]
1,I,I
2,"INTRODUCTION\nThis appeal centers on the application of the notice requirement of the Differing Site Conditions clause found in a contract between appellant Neal & Company, Inc","INTRODUCTION This appeal centers on the application of the notice requirement of the [MASK] Site [MASK] clause found in a contract [MASK] appellant Neal & Company, Inc"
3,(NCI) and appellee City of Dillingham (City),(NCI) and appellee City of Dillingham (City)
4,"Ap-pellee CH2M Hill (Hill), the City's engineer on the project, is involved in this appeal primarily because it acted as the City's representative on the project","Ap-pellee CH2M Hill (Hill), the City's engineer on the project, [MASK] involved in this appeal primarily because it [MASK] as the City's representative on [MASK] project"
5,"NCI claims that it encountered difficulties in excavation during the project because of unexpected soil conditions, that it gave notice of these unexpected conditions to Hill (and thus constructively to the City), and that therefore it is entitled to assert a claim under the Differing Site Conditions clause","NCI claims [MASK] it encountered difficulties in excavation [MASK] the project because of unexpected soil conditions, that it gave notice of these [MASK] conditions to Hill [MASK] thus constructively to the City), and that therefore it is entitled to assert a [MASK] under the [MASK] Site Conditions clause"
6,The City and Hill claim that no notice of a differing site condition was given,The [MASK] [MASK] Hill claim that no notice of a differing site condition was given
7,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition
8,"NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge","NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge"
9,II,II


In [ ]:
import pandas as pd
from transformers import pipeline, BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [ ]:
# Load the fill-mask pipeline and BERT model
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [ ]:
from transformers import BertTokenizer, BertModel
import torch

# Function to calculate sentence embeddings from BERT
def get_bert_embedding(sentence):
    # Load pre-trained BERT model and tokenizer
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertModel.from_pretrained('bert-base-uncased')  # Ensure you use BertModel, not BertForMaskedLM

    # Tokenize the input sentence
    inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True)

    # Get BERT model outputs (last hidden state)
    with torch.no_grad():  # Disable gradient calculation for efficiency
        outputs = model(**inputs)

    # Get the embeddings of the [CLS] token (first token) as the sentence embedding
    cls_embedding = outputs.last_hidden_state[:, 0, :]

    return cls_embedding


In [ ]:
# Function to compute semantic similarity between two sentences
def get_semantic_similarity(original_text, filled_text):
    original_embedding = get_bert_embedding(original_text)
    filled_embedding = get_bert_embedding(filled_text)

    # Compute cosine similarity between the two embeddings
    similarity = cosine_similarity(original_embedding.numpy(), filled_embedding.numpy())[0][0]
    return similarity


In [ ]:
df_sentences = df_sentences.head(100)

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import pandas as pd
from tqdm import tqdm

# Initialize tqdm for pandas
tqdm.pandas()

# Define the model name
model_name = "bert-base-uncased"  # You can replace this with other models like roberta-base, etc.

# Load pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

# Function to fill masked sentences
def predict_masked_sentence(masked_sentence):
    # Tokenize input text
    inputs = tokenizer(masked_sentence, return_tensors="pt")

    # Find masked token positions
    mask_token_index = torch.where(inputs.input_ids == tokenizer.mask_token_id)[1]

    # Forward pass to get predictions
    with torch.no_grad():
        logits = model(**inputs).logits

    # Decode the predictions for the masked tokens
    predicted_tokens = []
    for mask_index in mask_token_index:
        predicted_token_id = logits[0, mask_index].argmax(axis=-1)  # Get the most likely token
        predicted_token = tokenizer.decode(predicted_token_id)
        predicted_tokens.append(predicted_token)

    # Replace the masked tokens with the predicted tokens
    filled_text = masked_sentence
    for mask_token, predicted_token in zip(mask_token_index, predicted_tokens):
        filled_text = filled_text.replace("[MASK]", predicted_token, 1)

    return filled_text

# Assuming df_sentences is your DataFrame with shape (1000, 1)
# Apply the prediction function with tqdm progress bar
df_sentences['predicted_sentences'] = df_sentences['masked_sentences'].progress_apply(predict_masked_sentence)

# Now df_sentences contains a new column 'predicted_sentences' with the predicted masked sentences


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
100%|██████████| 100/100 [00:18<00:00,  5.34it/s]


In [ ]:
df_sentences

,sentence,masked_sentences,predicted_sentences
0,"OPINION\nRABINOWITZ, Justice",OPINION [MASK] [MASK],OPINION polls .
1,I,I,I
2,"INTRODUCTION\nThis appeal centers on the application of the notice requirement of the Differing Site Conditions clause found in a contract between appellant Neal & Company, Inc","INTRODUCTION This appeal centers on the application of the notice requirement of the [MASK] Site [MASK] clause found in a contract [MASK] appellant Neal & Company, Inc","INTRODUCTION This appeal centers on the application of the notice requirement of the "" Site "" clause found in a contract between appellant Neal & Company, Inc"
3,(NCI) and appellee City of Dillingham (City),(NCI) and appellee City of Dillingham (City),(NCI) and appellee City of Dillingham (City)
4,"Ap-pellee CH2M Hill (Hill), the City's engineer on the project, is involved in this appeal primarily because it acted as the City's representative on the project","Ap-pellee CH2M Hill (Hill), the City's engineer on the project, [MASK] involved in this appeal primarily because it [MASK] as the City's representative on [MASK] project","Ap-pellee CH2M Hill (Hill), the City's engineer on the project, was involved in this appeal primarily because it served as the City's representative on the project"
5,"NCI claims that it encountered difficulties in excavation during the project because of unexpected soil conditions, that it gave notice of these unexpected conditions to Hill (and thus constructively to the City), and that therefore it is entitled to assert a claim under the Differing Site Conditions clause","NCI claims [MASK] it encountered difficulties in excavation [MASK] the project because of unexpected soil conditions, that it gave notice of these [MASK] conditions to Hill [MASK] thus constructively to the City), and that therefore it is entitled to assert a [MASK] under the [MASK] Site Conditions clause","NCI claims that it encountered difficulties in excavation of the project because of unexpected soil conditions, that it gave notice of these soil conditions to Hill ( thus constructively to the City), and that therefore it is entitled to assert a claim under the appropriate Site Conditions clause"
6,The City and Hill claim that no notice of a differing site condition was given,The [MASK] [MASK] Hill claim that no notice of a differing site condition was given,The nearby and Hill claim that no notice of a differing site condition was given
7,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition
8,"NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge","NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge","NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge"
9,II,II,II


In [ ]:
v1 = "big dog"
v2 = "big cat"
print(get_semantic_similarity(v1, v2))
df_sentences['semantic similarity'] = df_sentences.progress_apply(
    lambda row: get_semantic_similarity(row['predicted_sentences'], row['sentence']), axis=1
)


0.9618647


100%|██████████| 100/100 [02:01<00:00,  1.21s/it]


In [ ]:
df_sentences.head(100)

,sentence,masked_sentences,predicted_sentences,semantic similarity
0,"OPINION\nRABINOWITZ, Justice",OPINION [MASK] [MASK],OPINION polls .,0.684500
1,I,I,I,1.000000
2,"INTRODUCTION\nThis appeal centers on the application of the notice requirement of the Differing Site Conditions clause found in a contract between appellant Neal & Company, Inc","INTRODUCTION This appeal centers on the application of the notice requirement of the [MASK] Site [MASK] clause found in a contract [MASK] appellant Neal & Company, Inc","INTRODUCTION This appeal centers on the application of the notice requirement of the "" Site "" clause found in a contract between appellant Neal & Company, Inc",0.996927
3,(NCI) and appellee City of Dillingham (City),(NCI) and appellee City of Dillingham (City),(NCI) and appellee City of Dillingham (City),1.000000
4,"Ap-pellee CH2M Hill (Hill), the City's engineer on the project, is involved in this appeal primarily because it acted as the City's representative on the project","Ap-pellee CH2M Hill (Hill), the City's engineer on the project, [MASK] involved in this appeal primarily because it [MASK] as the City's representative on [MASK] project","Ap-pellee CH2M Hill (Hill), the City's engineer on the project, was involved in this appeal primarily because it served as the City's representative on the project",0.984452
5,"NCI claims that it encountered difficulties in excavation during the project because of unexpected soil conditions, that it gave notice of these unexpected conditions to Hill (and thus constructively to the City), and that therefore it is entitled to assert a claim under the Differing Site Conditions clause","NCI claims [MASK] it encountered difficulties in excavation [MASK] the project because of unexpected soil conditions, that it gave notice of these [MASK] conditions to Hill [MASK] thus constructively to the City), and that therefore it is entitled to assert a [MASK] under the [MASK] Site Conditions clause","NCI claims that it encountered difficulties in excavation of the project because of unexpected soil conditions, that it gave notice of these soil conditions to Hill ( thus constructively to the City), and that therefore it is entitled to assert a claim under the appropriate Site Conditions clause",0.991062
6,The City and Hill claim that no notice of a differing site condition was given,The [MASK] [MASK] Hill claim that no notice of a differing site condition was given,The nearby and Hill claim that no notice of a differing site condition was given,0.965059
7,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,The superior court ruled on partial summary judgment that NCI did not give adequate notice of a differing site condition,1.000000
8,"NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge","NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge","NCI also appeals the superior court's denial of its motions for leave to amend its complaint, for continuance of the trial, and for disqualification of thé trial judge",1.000000
9,II,II,II,1.000000


In [ ]:
df_sentences['semantic similarity'].mean()

0.9793442